# 00 · Setup check

**Building and Fine-Tuning Decision Models for AI Agents** · To Data & Beyond

Run every cell. Each check prints one line: `OK`, `MISSING` or `INFO`. Post `OK` in the chat,
or paste the first line that is not OK.

| Mode | What runs | Keys |
|---|---|---|
| Replay | Saved Jev and LLM answers from the recorded runs. Every notebook works. | none |
| Live | New calls to the Jev API (and optionally an LLM baseline) | `TYPESAFE_API_KEY` (+ `OPENAI_API_KEY`) |
| Fine-tuning | Training Qwen3 on a GPU and publishing it | `HF_TOKEN` with **write** access, `HF_USERNAME` |

**Where to put keys.** Locally: copy `.env.example` to `.env` and fill it in.
In Colab: click the key icon in the left sidebar (Secrets) and add the same names.
The notebooks never print a key.

## 1 · Get the workshop files (Colab only)

Locally, open the notebooks from the workshop folder and skip this cell.
In Colab, set `REPO_URL` to the workshop repository, or leave it empty to upload the workshop zip.

In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/To-Data-Beyond/Building-and-Fine-Tuning-Decision-Models-for-AI-Agents.git"   # the workshop GitHub repository
TARGET = Path("/content/decision-models-workshop")

if "google.colab" in sys.modules and not TARGET.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(TARGET)], check=True)
    else:
        from google.colab import files
        print("Upload the workshop zip file:")
        up = files.upload()
        zip_name = next(iter(up))
        subprocess.run(["unzip", "-q", zip_name, "-d", "/content"], check=True)
        found = next(Path("/content").glob("**/dmw/__init__.py")).parents[1]
        if found != TARGET:
            found.rename(TARGET)
    os.chdir(TARGET)
print("Working folder:", Path.cwd())

## 2 · Install packages

In [ ]:
# Core packages for every notebook (quick). Fine-tuning packages are installed in notebook 07.
%pip install -q -r requirements.txt

## 3 · Checks

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import importlib, platform, shutil, socket

def line(status, msg): print(f"{status:<8}{msg}")

v = sys.version_info
line("OK" if v >= (3, 10) else "MISSING", f"Python {platform.python_version()} (need 3.10+)")
for pkg in ["pandas", "matplotlib", "certifi"]:
    try:
        importlib.import_module(pkg); line("OK", f"package {pkg}")
    except ImportError:
        line("MISSING", f"package {pkg}: run the install cell above")

try:
    import torch
    dev = config.device()
    detail = torch.cuda.get_device_name(0) if dev == "cuda" else dev
    line("OK" if dev != "cpu" else "INFO", f"torch {torch.__version__}, accelerator: {detail}"
         + ("" if dev != "cpu" else "  (fine for notebooks 01-05; use a GPU runtime for 07)"))
except ImportError:
    line("INFO", "torch not installed yet: notebook 06/07 installs it")

free_gb = shutil.disk_usage(ROOT).free / 1e9
line("OK" if free_gb > 10 else "INFO", f"free disk {free_gb:.0f} GB (fine-tuning needs ~10 GB)")

for name, why in [("TYPESAFE_API_KEY", "live Jev calls (optional)"), ("OPENAI_API_KEY", "LLM baseline (optional)"),
                  ("HF_TOKEN", "publishing your model"), ("HF_USERNAME", "your Hub repo name")]:
    line("OK" if config.key(name) else ("MISSING" if name.startswith("HF") else "INFO"),
         f"{name} {'set' if config.key(name) else 'not set'}: {why}")

def reachable(host):
    try:
        socket.create_connection((host, 443), timeout=5).close(); return True
    except OSError:
        return False
for host in ["api.typesafe.ai", "huggingface.co"]:
    line("OK" if reachable(host) else "MISSING", f"network: {host}")

print("\nRun mode for the Jev notebooks:", config.mode())

In [ ]:
# Optional: confirm the Hugging Face token can write (needed in notebook 07)
if config.key("HF_TOKEN"):
    %pip install -q huggingface_hub
    from huggingface_hub import whoami
    me = whoami(token=config.key("HF_TOKEN"))
    role = me.get("auth", {}).get("accessToken", {}).get("role")
    print("Logged in to Hugging Face as", me["name"], "| token role:", role)
    if config.key("HF_USERNAME") and config.key("HF_USERNAME") != me["name"]:
        print("Note: HF_USERNAME differs from the token owner; pushes go to HF_USERNAME, so you need write access there.")
else:
    print("HF_TOKEN not set yet. Create one at huggingface.co → Settings → Access Tokens → Write.")